# 📚 Proposition Chunking — Fully Local with Ollama

**Model:** `phi4-mini` (LLM) + `nomic-embed-text:v1.5` (embeddings)  
**No API keys needed. Everything runs locally via Ollama.**

### What This Notebook Does
Implements **Proposition Chunking** — a RAG technique from [Chen et al. (2023)](https://doi.org/10.48550/arXiv.2312.06648) that breaks documents into atomic, self-contained factual statements instead of raw text chunks.

**Pipeline:**
1. Split document into chunks
2. Use `phi4-mini` to generate propositions from each chunk
3. Use `phi4-mini` to quality-check each proposition (accuracy / clarity / completeness / conciseness)
4. Embed passing propositions into FAISS
5. Compare retrieval quality vs. naive large-chunk retrieval

---
### ⚠️ Prerequisites — Run Before Starting
```bash
# 1. Install Ollama: https://ollama.com/download
# 2. Pull the models
ollama pull phi4-mini
ollama pull nomic-embed-text:v1.5
# 3. Make sure Ollama is running
ollama serve
```

## Cell 1 — Install Dependencies

In [1]:
!pip install faiss-cpu langchain langchain-community langchain-ollama langchain-core

## Cell 2 — Imports & Model Setup

In [2]:
from typing import List, Optional
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pydantic import BaseModel, Field
import json

# ── Models ──────────────────────────────────────────────────────────────────
LLM_MODEL       = "phi4-mini"           # local LLM for proposition generation & grading
EMBED_MODEL     = "nomic-embed-text:v1.5"  # local embedding model

llm = ChatOllama(
    model=LLM_MODEL,
    temperature=0,          # deterministic — same input → same propositions every run
)

embedding_model = OllamaEmbeddings(
    model=EMBED_MODEL,
)

print(f"✅ LLM      : {LLM_MODEL}")
print(f"✅ Embeddings: {EMBED_MODEL}")

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'


✅ LLM      : phi4-mini
✅ Embeddings: nomic-embed-text:v1.5


## Cell 3 — Sample Document

In [3]:
sample_content = """Paul Graham's essay \"Founder Mode,\" published in September 2024, challenges conventional wisdom about scaling startups, arguing that founders should maintain their unique management style rather than adopting traditional corporate practices as their companies grow.
Conventional Wisdom vs. Founder Mode
The essay argues that the traditional advice given to growing companies—hiring good people and giving them autonomy—often fails when applied to startups.
This approach, suitable for established companies, can be detrimental to startups where the founder's vision and direct involvement are crucial. \"Founder Mode\" is presented as an emerging paradigm that is not yet fully understood or documented, contrasting with the conventional \"manager mode\" often advised by business schools and professional managers.
Unique Founder Abilities
Founders possess unique insights and abilities that professional managers do not, primarily because they have a deep understanding of their company's vision and culture.
Graham suggests that founders should leverage these strengths rather than conform to traditional managerial practices. \"Founder Mode\" is an emerging paradigm that is not yet fully understood or documented, with Graham hoping that over time, it will become as well-understood as the traditional manager mode, allowing founders to maintain their unique approach even as their companies scale.
Challenges of Scaling Startups
As startups grow, there is a common belief that they must transition to a more structured managerial approach. However, many founders have found this transition problematic, as it often leads to a loss of the innovative and agile spirit that drove the startup's initial success.
Brian Chesky, co-founder of Airbnb, shared his experience of being advised to run the company in a traditional managerial style, which led to poor outcomes. He eventually found success by adopting a different approach, influenced by how Steve Jobs managed Apple.
Steve Jobs' Management Style
Steve Jobs' management approach at Apple served as inspiration for Brian Chesky's \"Founder Mode\" at Airbnb. One notable practice was Jobs' annual retreat for the 100 most important people at Apple, regardless of their position on the organizational chart. This unconventional method allowed Jobs to maintain a startup-like environment even as Apple grew, fostering innovation and direct communication across hierarchical levels. Such practices emphasize the importance of founders staying deeply involved in their companies' operations, challenging the traditional notion of delegating responsibilities to professional managers as companies scale.
"""

docs_list = [Document(
    page_content=sample_content,
    metadata={
        "Title": "Paul Graham's Founder Mode Essay",
        "Source": "https://www.perplexity.ai/page/paul-graham-s-founder-mode-ess-t9TCyvkqRiyMQJWsHr0fnQ"
    }
)]

print(f"📄 Document loaded — {len(sample_content)} characters")

📄 Document loaded — 2648 characters


## Cell 4 — Chunk the Document

In [4]:
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=200,
    chunk_overlap=50,
)

doc_splits = text_splitter.split_documents(docs_list)

# Tag each chunk with its index (used later to trace which chunk a proposition came from)
for i, doc in enumerate(doc_splits):
    doc.metadata["chunk_id"] = i + 1

print(f" Split into {len(doc_splits)} chunks")
for i, chunk in enumerate(doc_splits):
    print(f"\n--- Chunk {i+1} ({len(chunk.page_content)} chars) ---")
    print(chunk.page_content[:200], "..." if len(chunk.page_content) > 200 else "")

 Split into 3 chunks

--- Chunk 1 (1006 chars) ---
Paul Graham's essay "Founder Mode," published in September 2024, challenges conventional wisdom about scaling startups, arguing that founders should maintain their unique management style rather than  ...

--- Chunk 2 (895 chars) ---
Unique Founder Abilities
Founders possess unique insights and abilities that professional managers do not, primarily because they have a deep understanding of their company's vision and culture.
Graha ...

--- Chunk 3 (939 chars) ---
Brian Chesky, co-founder of Airbnb, shared his experience of being advised to run the company in a traditional managerial style, which led to poor outcomes. He eventually found success by adopting a d ...


## Cell 5 — Proposition Generation (Structured Output via JSON)

In [5]:
# ── Pydantic schema ──────────────────────────────────────────────────────────
class GeneratePropositions(BaseModel):
    """List of all propositions extracted from a document chunk."""
    propositions: List[str] = Field(
        description="List of factual, self-contained, atomic propositions"
    )


# ── Few-shot examples ────────────────────────────────────────────────────────
proposition_examples = [
    {
        "document": "In 1969, Neil Armstrong became the first person to walk on the Moon during the Apollo 11 mission.",
        "propositions": json.dumps([
            "Neil Armstrong was an astronaut.",
            "Neil Armstrong walked on the Moon in 1969.",
            "Neil Armstrong was the first person to walk on the Moon.",
            "Neil Armstrong walked on the Moon during the Apollo 11 mission.",
            "The Apollo 11 mission occurred in 1969."
        ])
    },
]

example_prompt = ChatPromptTemplate.from_messages([
    ("human",  "{document}"),
    ("ai",     "{propositions}"),
])

few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=proposition_examples,
)

# ── System prompt ────────────────────────────────────────────────────────────
PROPOSITION_SYSTEM = """Break the text into simple, self-contained propositions.
Each proposition must:
1. Express exactly ONE fact or claim.
2. Be understandable with NO additional context.
3. Use full names — no pronouns or ambiguous references.
4. Include relevant dates or qualifiers where applicable.
5. Contain ONE subject-predicate relationship — no conjunctions.

Reply ONLY with a valid JSON object in this exact format:
{"propositions": ["fact one.", "fact two.", ...]}
No extra text, no markdown, no explanation."""

proposition_prompt = ChatPromptTemplate.from_messages([
    ("system", PROPOSITION_SYSTEM),
    few_shot_prompt,
    ("human",  "{document}"),
])

# ── Parser: phi4-mini may not support .with_structured_output natively ───────
# We'll use a manual JSON parse fallback for robustness
def parse_propositions(text: str) -> List[str]:
    """Extract propositions list from LLM text output."""
    text = text.strip()
    # Try direct JSON parse
    try:
        data = json.loads(text)
        if isinstance(data, dict) and "propositions" in data:
            return data["propositions"]
        if isinstance(data, list):
            return data
    except json.JSONDecodeError:
        pass
    # Try to find JSON block inside markdown fences
    import re
    match = re.search(r'\{.*?"propositions".*?\}', text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group(0))["propositions"]
        except Exception:
            pass
    # Last resort: bullet / line-split
    lines = [l.strip().lstrip("•-*0123456789.) ") for l in text.splitlines() if l.strip()]
    return [l for l in lines if len(l) > 10]


proposition_chain = proposition_prompt | llm

print("Proposition generation chain ready")

Proposition generation chain ready


## Cell 6 — Generate Propositions from Each Chunk

In [12]:
PROPOSITION_SYSTEM = """Break the text into simple, self-contained propositions.
Each proposition must:
1. Express exactly ONE fact or claim.
2. Be understandable with NO additional context.
3. Use full names, no pronouns or ambiguous references.
4. Include relevant dates or qualifiers where applicable.
5. Contain ONE subject-predicate relationship, no conjunctions.

Reply ONLY with valid JSON, no markdown, no explanation:
{{"propositions": ["fact one.", "fact two.", ...]}}

Example
-------
Input: In 1969, Neil Armstrong became the first person to walk on the Moon during the Apollo 11 mission.
Output: {{"propositions": ["Neil Armstrong was an astronaut.", "Neil Armstrong walked on the Moon in 1969.", "Neil Armstrong was the first person to walk on the Moon.", "Neil Armstrong walked on the Moon during the Apollo 11 mission.", "The Apollo 11 mission occurred in 1969."]}}"""

proposition_prompt = ChatPromptTemplate.from_messages([
    ("system", PROPOSITION_SYSTEM),
    ("human",  "{document}"),
])

proposition_chain = proposition_prompt | llm
print("Chain ready — input variables:", proposition_prompt.input_variables)
# Should print: ['document']

Chain ready — input variables: ['document']


In [17]:
propositions: list = []

for i, chunk in enumerate(doc_splits):
    print(f"\nProcessing chunk {i+1}/{len(doc_splits)}...")
    response = proposition_chain.invoke({"document": chunk.page_content})
    raw_text = response.content if hasattr(response, "content") else str(response)
    extracted = parse_propositions(raw_text)

    # Flatten in case model returned a list of lists
    flat = []
    for item in extracted:
        if isinstance(item, list):
            flat.extend(item)
        elif isinstance(item, str):
            flat.append(item)

    print(f"   ↳ {len(flat)} propositions extracted")
    for prop in flat:
        if prop.strip():
            propositions.append(Document(
                page_content=prop.strip(),
                metadata={
                    "Title":    "Paul Graham's Founder Mode Essay",
                    "Source":   "https://www.perplexity.ai/page/paul-graham-s-founder-mode-ess-t9TCyvkqRiyMQJWsHr0fnQ",
                    "chunk_id": i + 1,
                }
            ))

print(f"\nTotal propositions generated: {len(propositions)}")
for p in propositions[:5]:
    print(f"  • {p.page_content}")


Processing chunk 1/3...
   ↳ 7 propositions extracted

Processing chunk 2/3...
   ↳ 7 propositions extracted

Processing chunk 3/3...
   ↳ 5 propositions extracted

Total propositions generated: 19
  • Paul Graham's essay 'Founder Mode' was published in September 2024.
  • The conventional wisdom about scaling startups challenges the idea presented in Paul Graham's essay 'Founder Mode'.
  • Conventional advice for growing companies includes hiring good people with autonomy, which is often ineffective according to Paul's article.
  • Traditional corporate practices are not recommended by Paul Graham as founders grow their businesses.
  • Paul Graham argues that founder mode should be maintained rather than adopting traditional management styles in startups.


## Cell 7 — Quality-Check Schema & Grader Chain

In [20]:
GRADER_SYSTEM = """Evaluate the proposition against the original text chunk.
Score each dimension 1-10.

Reply ONLY with valid JSON, no markdown, no explanation:
{{"accuracy": <int>, "clarity": <int>, "completeness": <int>, "conciseness": <int>}}"""

GRADER_HUMAN = """Original text: {original_text}

Proposition: {proposition}"""

grader_prompt = ChatPromptTemplate.from_messages([
    ("system", GRADER_SYSTEM),
    ("human",  GRADER_HUMAN),
])

grader_chain = grader_prompt | llm

# Verify — should only show: ['original_text', 'proposition']
print("Grader input variables:", grader_prompt.input_variables)

Grader input variables: ['original_text', 'proposition']


In [21]:
# Debug cell — run this before the full loop
test_prop = propositions[0]
raw = grader_chain.invoke({
    "proposition": test_prop.page_content,
    "original_text": doc_splits[test_prop.metadata["chunk_id"] - 1].page_content
})
print(repr(raw.content))   # see exactly what phi4-mini returns

'```json\n{"accuracy": 10, "clarity": 9, "completeness": 8, "conciseness": 7}\n```'


In [24]:
def parse_grades(text: str) -> dict:
    """Parse JSON scores from LLM output with fallback."""
    import re
    text = text.strip()

    # Strip markdown fences if present
    text = re.sub(r"```(?:json)?", "", text).strip()

    # 1. Direct JSON parse
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    # 2. Find first {...} block containing "accuracy"
    match = re.search(r'\{[^{}]*"accuracy"[^{}]*\}', text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group(0))
        except Exception:
            pass

    # 3. Extract individual scores with regex (handles "accuracy: 8" prose output)
    scores = {}
    for key in ["accuracy", "clarity", "completeness", "conciseness"]:
        m = re.search(rf'"{key}"\s*:\s*(\d+)', text)
        if not m:
            m = re.search(rf'{key}\s*[:\-]\s*(\d+)', text, re.IGNORECASE)
        if m:
            scores[key] = int(m.group(1))

    if len(scores) == 4:
        return scores

    # 4. Could not parse — print raw output so you can see what the model returned
    print(f"   ⚠️  Could not parse grades. Raw output:\n{text[:300]}\n")
    return {"accuracy": 6, "clarity": 6, "completeness": 6, "conciseness": 6}

## Cell 8 — Run Quality Check on Every Proposition

In [27]:
THRESHOLDS = {"accuracy": 7, "clarity": 7, "completeness": 7, "conciseness": 6}

def passes_quality_check(scores: dict) -> bool:
    return all(scores.get(k, 0) >= v for k, v in THRESHOLDS.items())

evaluated_propositions: List[Document] = []
failed = []

for idx, prop_doc in enumerate(propositions):
    chunk_id      = prop_doc.metadata["chunk_id"]
    original_text = doc_splits[chunk_id - 1].page_content
    response  = grader_chain.invoke({
        "proposition":   prop_doc.page_content,
        "original_text": original_text,
    })
    raw_text = response.content if hasattr(response, "content") else str(response)
    scores   = parse_grades(raw_text)

    if passes_quality_check(scores):
        evaluated_propositions.append(prop_doc)
    else:
        failed.append((idx + 1, prop_doc.page_content, scores))
        print(f"  ❌ FAIL  [{idx+1}] {prop_doc.page_content}")
        print(f"         Scores: {scores}")

print(f"\n✅ Passed: {len(evaluated_propositions)} / {len(propositions)} propositions")
print(f"❌ Failed: {len(failed)}")

  ❌ FAIL  [2] The conventional wisdom about scaling startups challenges the idea presented in Paul Graham's essay 'Founder Mode'.
         Scores: {'accuracy': 2, 'clarity': 5, 'completeness': 3, 'conciseness': 4}
  ❌ FAIL  [9] Professional managers do not share these unique insights.
         Scores: {'accuracy': 8, 'clarity': 7, 'completeness': 6, 'conciseness': 5}
  ❌ FAIL  [13] Traditional manager mode has been fully understood for a long period.
         Scores: {'accuracy': 2, 'clarity': 7, 'completeness': 4, 'conciseness': 6}

✅ Passed: 16 / 19 propositions
❌ Failed: 3


## Cell 9 — Embed Propositions into FAISS

In [30]:
print(f" Embedding {len(evaluated_propositions)} propositions...")

vectorstore_propositions = FAISS.from_documents(evaluated_propositions, embedding_model)

retriever_propositions = vectorstore_propositions.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4},
)

print(" Proposition vectorstore ready")

 Embedding 16 propositions...
 Proposition vectorstore ready


## Cell 10 — Build Larger-Chunk Retriever (Baseline)

In [31]:
print(f" Embedding {len(doc_splits)} larger chunks (baseline)...")

vectorstore_larger = FAISS.from_documents(doc_splits, embedding_model)

retriever_larger = vectorstore_larger.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4},
)

print("Larger-chunk vectorstore ready")

 Embedding 3 larger chunks (baseline)...
Larger-chunk vectorstore ready


## Cell 11 — Helper: Pretty-Print Results Side by Side

In [32]:
def print_results(query: str, res_prop: list, res_large: list):
    bar = "─" * 70
    print(f"\n{'='*70}")
    print(f"QUERY: {query}")
    print(f"{'='*70}")

    print(f"\nPROPOSITION-BASED RETRIEVAL")
    print(bar)
    for i, r in enumerate(res_prop):
        print(f"{i+1}. [{r.metadata['chunk_id']}] {r.page_content}")

    print(f"\nLARGER-CHUNK RETRIEVAL (Baseline)")
    print(bar)
    for i, r in enumerate(res_large):
        preview = r.page_content[:180].replace("\n", " ")
        print(f"{i+1}. [chunk {r.metadata['chunk_id']}] {preview}...")

print("✅ Helper ready")

✅ Helper ready


## Cell 12 — Test Query: Steve Jobs / Airbnb

In [33]:
query = "Who's management approach served as inspiration for Brian Chesky's 'Founder Mode' at Airbnb?"

res_prop  = retriever_propositions.invoke(query)
res_large = retriever_larger.invoke(query)

print_results(query, res_prop, res_large)


QUERY: Who's management approach served as inspiration for Brian Chesky's 'Founder Mode' at Airbnb?

PROPOSITION-BASED RETRIEVAL
──────────────────────────────────────────────────────────────────────
1. [3] Steve Jobs managed Apple with a unique approach that inspired Brian Chesky's 'Founder Mode'.
2. [3] Traditional managerial style led to poor outcomes for Brian Chesky at Airbnb.
3. [1] The 'Founder Mode' contrasts the conventional wisdom of business schools and professional managers about scaling companies.
4. [2] Graham suggests leveraging founder strengths rather than conforming to traditional managerial practices.

LARGER-CHUNK RETRIEVAL (Baseline)
──────────────────────────────────────────────────────────────────────
1. [chunk 3] Brian Chesky, co-founder of Airbnb, shared his experience of being advised to run the company in a traditional managerial style, which led to poor outcomes. He eventually found suc...
2. [chunk 2] Unique Founder Abilities Founders possess unique insigh

## Cell 13 — Test 1: What Is the Essay About?

In [34]:
q1 = 'What is the essay "Founder Mode" about?'
print_results(q1, retriever_propositions.invoke(q1), retriever_larger.invoke(q1))


QUERY: What is the essay "Founder Mode" about?

PROPOSITION-BASED RETRIEVAL
──────────────────────────────────────────────────────────────────────
1. [1] Paul Graham's essay 'Founder Mode' was published in September 2024.
2. [1] Founders have unique insights due to a deep understanding of company vision, according to Paul's essay.
3. [1] The 'Founder Mode' contrasts the conventional wisdom of business schools and professional managers about scaling companies.
4. [2] Founder Mode is an emerging paradigm that Graham hopes will become well-understood over time.

LARGER-CHUNK RETRIEVAL (Baseline)
──────────────────────────────────────────────────────────────────────
1. [chunk 1] Paul Graham's essay "Founder Mode," published in September 2024, challenges conventional wisdom about scaling startups, arguing that founders should maintain their unique managemen...
2. [chunk 2] Unique Founder Abilities Founders possess unique insights and abilities that professional managers do not, primarily b

## Cell 14 — Test 2: Co-Founder of Airbnb?

In [35]:
q2 = "Who is the co-founder of Airbnb?"
print_results(q2, retriever_propositions.invoke(q2), retriever_larger.invoke(q2))


QUERY: Who is the co-founder of Airbnb?

PROPOSITION-BASED RETRIEVAL
──────────────────────────────────────────────────────────────────────
1. [3] Brian Chesky co-founded Airbnb.
2. [3] Steve Jobs managed Apple with a unique approach that inspired Brian Chesky's 'Founder Mode'.
3. [2] Many founders find transitioning from Founder Mode challenging as their companies scale.
4. [1] Paul Graham argues that founder mode should be maintained rather than adopting traditional management styles in startups.

LARGER-CHUNK RETRIEVAL (Baseline)
──────────────────────────────────────────────────────────────────────
1. [chunk 3] Brian Chesky, co-founder of Airbnb, shared his experience of being advised to run the company in a traditional managerial style, which led to poor outcomes. He eventually found suc...
2. [chunk 2] Unique Founder Abilities Founders possess unique insights and abilities that professional managers do not, primarily because they have a deep understanding of their company's visi

## Cell 15 — Test 3: When Was the Essay Published?

In [36]:
q3 = 'When was the essay "Founder Mode" published?'
print_results(q3, retriever_propositions.invoke(q3), retriever_larger.invoke(q3))


QUERY: When was the essay "Founder Mode" published?

PROPOSITION-BASED RETRIEVAL
──────────────────────────────────────────────────────────────────────
1. [1] Paul Graham's essay 'Founder Mode' was published in September 2024.
2. [1] Founders have unique insights due to a deep understanding of company vision, according to Paul's essay.
3. [2] Founder Mode is an emerging paradigm that Graham hopes will become well-understood over time.
4. [1] The 'Founder Mode' contrasts the conventional wisdom of business schools and professional managers about scaling companies.

LARGER-CHUNK RETRIEVAL (Baseline)
──────────────────────────────────────────────────────────────────────
1. [chunk 1] Paul Graham's essay "Founder Mode," published in September 2024, challenges conventional wisdom about scaling startups, arguing that founders should maintain their unique managemen...
2. [chunk 3] Brian Chesky, co-founder of Airbnb, shared his experience of being advised to run the company in a traditional ma

## Cell 16 — Final Comparison Summary

| Aspect | Proposition-Based | Large-Chunk Baseline |
|---|---|---|
| **Precision** | ✅ High — one fact per result | ⚠️ Medium — relevant chunk but noisy |
| **Clarity** | ✅ Atomic, scannable | ⚠️ Requires reading the full paragraph |
| **Context** | ❌ Low — no surrounding narrative | ✅ High — full context preserved |
| **Comprehensiveness** | ❌ May miss adjacent details | ✅ Broader picture included |
| **Information Overload** | ✅ Minimal | ⚠️ Risk of overwhelming the LLM |
| **Best For** | Factual lookups, QA, slot-filling | Summarisation, open-ended questions |
| **LLM Cost** | 🔴 Higher (proposition gen + grading) | 🟢 Lower (direct embed) |

**When to use proposition chunking:**
- Dense technical / legal / medical documents where exact facts matter
- QA systems where you need the single most precise sentence
- When retrieval noise causes hallucinations in your answers

**When to skip it:**
- Summarisation tasks that need broad context
- Tight latency budgets (proposition gen doubles your preprocessing time)
- Simple documents where standard chunking already works well